# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [1]:
import os
import sys

repo_name = "2026-HYU-AUE8088-PA2"
repo_path = f"/content/{repo_name}"

# 네 fork clone
if not os.path.exists(repo_path):
    !git clone https://github.com/Jieunn-Kim/2026-HYU-AUE8088-PA2.git "{repo_path}"

# 레포 루트 이동
%cd "{repo_path}"

%load_ext autoreload
%autoreload 2

# 의존성 설치
!pip install -q -r requirements.txt

Cloning into '/content/2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 44 (delta 8), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 57.32 KiB | 962.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jieunnkim (jieunnkim-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=b0ae8666-62ce-403b-bf47-48628a057691
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:03<00:00, 78.9MB/s]


압축 해제 중...
완료 → ../data/set_a


In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
import os
import shutil

src_path = "/content/2026-HYU-AUE8088-PA2/src/models/vit.py"
backup_dir = "/content/drive/MyDrive/AUE8088_PA2/source_backup"
backup_path = os.path.join(backup_dir, "vit.py")

os.makedirs(backup_dir, exist_ok=True)
shutil.copy2(src_path, backup_path)

print("백업 완료:", backup_path)
print("파일 존재:", os.path.exists(backup_path))
print("파일 크기:", os.path.getsize(backup_path), "bytes")

백업 완료: /content/drive/MyDrive/AUE8088_PA2/source_backup/vit.py
파일 존재: True
파일 크기: 5191 bytes


scract 먼저

In [10]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
USE_PRETRAINED = False
model = vit_small_patch16_224().to(device)

In [11]:
epochs = 25

optim = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=5e-2,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=epochs,
)

losses = {
    a: nn.CrossEntropyLoss()
    for a in ATTRIBUTES
}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name="level2-vit_s16",
    config={
        "backbone": "vit_s16",
        "pretrained": False,
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16", "scratch"],
)

trainer = MultiTaskTrainer(
    model,
    optim,
    sched,
    losses,
    device,
    TrainConfig(epochs=epochs),
    logger=logger,
)

# 학습
history = trainer.fit(train_loader, val_loader)

# 최종 validation 예측
val_pred, _, val_tgt, _ = collect_predictions(
    model,
    val_loader,
    device,
)

# Confusion matrix 기록
cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

# Drive에 체크포인트 저장
save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "level2_vit_s16_scratch.pth",
)

torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "pretrained": False,
        "epochs": epochs,
    },
    save_path,
)

print("저장 완료:", save_path)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/25] train_loss=2.3439  val_avg_MF1=0.3678  per={'weather': 0.20932267884322678, 'scene': 0.2531017369727047, 'timeofday': 0.6410559495665878}


[epoch 02/25] train_loss=2.0771  val_avg_MF1=0.3916  per={'weather': 0.1563124821479577, 'scene': 0.35963880849011093, 'timeofday': 0.6589850731295143}


[epoch 03/25] train_loss=2.0338  val_avg_MF1=0.4050  per={'weather': 0.21968739632375692, 'scene': 0.31546930359080366, 'timeofday': 0.6798050960905769}


[epoch 04/25] train_loss=1.9997  val_avg_MF1=0.4089  per={'weather': 0.2289548827616178, 'scene': 0.3566172711064872, 'timeofday': 0.6411762801154993}


...

[epoch 23/25] train_loss=1.6423  val_avg_MF1=0.5319  per={'weather': 0.35751514316342003, 'scene': 0.39788412582671323, 'timeofday': 0.8404161124447764}


[epoch 24/25] train_loss=1.6375  val_avg_MF1=0.5363  per={'weather': 0.3569382913649796, 'scene': 0.4159215899361128, 'timeofday': 0.8360556291815083}


[epoch 25/25] train_loss=1.6199  val_avg_MF1=0.5363  per={'weather': 0.3569382913649796, 'scene': 0.4067018742880812, 'timeofday': 0.8451986505424447}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▅▅▅▅▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▁▁▁▁▁
val/avg_macro_f1,▁▂▃▃▄▄▅▄▄▅▆▅▇▆▇▇█▇▇██████
val/mf1_scene,▁▆▄▅▄▅▆▄▄▆█▆▇▆▇▇▇█▇▇▇▇▇██
val/mf1_timeofday,▁▂▂▁▄▄▄▄▄▃▅▄▇▆▆▆▇▇▇██████
val/mf1_weather,▃▁▃▃▄▄▆▅▆▆▆▅▆█▇▆▇▆▇██▇███
epoch,25
lr,0
train/loss,1.61989
val/avg_macro_f1,0.53628


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level2_vit_s16_scratch.pth


Pretrained 버전

In [12]:
# Pretrained ViT-S/16 모델 준비 셀
import os
import gc
import torch

# Scratch 학습 객체와 GPU 메모리 정리
for name in ["trainer", "optim", "sched", "losses", "model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

USE_PRETRAINED = True

PRETRAINED_SOURCE = "Facebook Research DeiT-S/16, ImageNet-1K"
PRETRAINED_DIR = "/content/drive/MyDrive/AUE8088_PA2/pretrained"

os.makedirs(PRETRAINED_DIR, exist_ok=True)

DEIT_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

# Drive에 없으면 다운로드하고, 있으면 기존 파일 재사용
checkpoint = torch.hub.load_state_dict_from_url(
    DEIT_URL,
    model_dir=PRETRAINED_DIR,
    map_location="cpu",
    check_hash=True,
)

pretrained_state = checkpoint["model"]

# 직접 구현한 ViT 생성
model = vit_small_patch16_224()
model_state = model.state_dict()

mapped_state = {}

for key, value in pretrained_state.items():
    new_key = key

    if new_key.startswith("module."):
        new_key = new_key[len("module."):]

    # DeiT MLP 이름을 현재 구현의 nn.Sequential 이름으로 변경
    new_key = new_key.replace(".mlp.fc1.", ".mlp.0.")
    new_key = new_key.replace(".mlp.fc2.", ".mlp.3.")

    # ImageNet 분류 head는 사용하지 않음
    if new_key.startswith("head."):
        continue

    # 이름과 크기가 모두 일치하는 가중치만 사용
    if (
        new_key in model_state
        and model_state[new_key].shape == value.shape
    ):
        mapped_state[new_key] = value

missing, unexpected = model.load_state_dict(
    mapped_state,
    strict=False,
)

print("Pretrained source:", PRETRAINED_SOURCE)
print("매칭된 pretrained tensor 수:", len(mapped_state))
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

if len(mapped_state) < 140:
    raise RuntimeError(
        "Pretrained 가중치 매칭 수가 너무 적습니다. "
        "vit.py의 레이어 이름을 확인하세요."
    )

model = model.to(device)

print("Pretrained ViT-S/16 준비 완료")

Downloading: "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth" to /content/drive/MyDrive/AUE8088_PA2/pretrained/deit_small_patch16_224-cd65a155.pth
100%|██████████| 84.2M/84.2M [00:01<00:00, 83.2MB/s]


Pretrained source: Facebook Research DeiT-S/16, ImageNet-1K
매칭된 pretrained tensor 수: 150
Missing keys: ['head.weather.weight', 'head.weather.bias', 'head.scene.weight', 'head.scene.bias', 'head.timeofday.weight', 'head.timeofday.bias']
Unexpected keys: []
Pretrained ViT-S/16 준비 완료


In [13]:
epochs = 25

optim = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=5e-2,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=epochs,
)

losses = {
    a: nn.CrossEntropyLoss()
    for a in ATTRIBUTES
}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name="level2-vit_s16-pretrained",
    config={
        "backbone": "vit_s16",
        "pretrained": True,
        "pretrained_source": PRETRAINED_SOURCE,
        "matched_pretrained_tensors": len(mapped_state),
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16", "pretrained"],
)

trainer = MultiTaskTrainer(
    model,
    optim,
    sched,
    losses,
    device,
    TrainConfig(epochs=epochs),
    logger=logger,
)

# 학습
history = trainer.fit(train_loader, val_loader)

# 최종 validation 예측 및 confusion matrix 기록
val_pred, _, val_tgt, _ = collect_predictions(
    model,
    val_loader,
    device,
)

cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

# Drive에 저장
save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "level2_vit_s16_pretrained.pth",
)

torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "pretrained": True,
        "pretrained_source": PRETRAINED_SOURCE,
        "matched_pretrained_tensors": len(mapped_state),
        "epochs": epochs,
    },
    save_path,
)

print("저장 완료:", save_path)

[epoch 01/25] train_loss=2.0101  val_avg_MF1=0.5006  per={'weather': 0.3909208658495502, 'scene': 0.44872835891427143, 'timeofday': 0.6620186053937798}


[epoch 02/25] train_loss=1.5937  val_avg_MF1=0.6273  per={'weather': 0.4829211937457192, 'scene': 0.5920554854981085, 'timeofday': 0.8068514861141588}


[epoch 03/25] train_loss=1.4562  val_avg_MF1=0.5974  per={'weather': 0.46361958078839655, 'scene': 0.5035230346542056, 'timeofday': 0.825126942478995}


[epoch 04/25] train_loss=1.3827  val_avg_MF1=0.6071  per={'weather': 0.41894776618537694, 'scene': 0.5727881054779617, 'timeofday': 0.8294762998480819}


[epoch 05/25] train_loss=1.3119  val_avg_MF1=0.6746  per={'weather': 0.5658609723610059, 'scene': 0.6461311549807125, 'timeofday': 0.8117112202218585}


...

[epoch 23/25] train_loss=0.0240  val_avg_MF1=0.6825  per={'weather': 0.580697492249974, 'scene': 0.6248728790589256, 'timeofday': 0.8418084035394479}


[epoch 24/25] train_loss=0.0223  val_avg_MF1=0.6875  per={'weather': 0.5784709834970775, 'scene': 0.6376258755252745, 'timeofday': 0.8462630517047879}


[epoch 25/25] train_loss=0.0244  val_avg_MF1=0.6860  per={'weather': 0.5759920744704677, 'scene': 0.6356640107915328, 'timeofday': 0.8462630517047879}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▆▅▅█▅▇█▇▆▇▇▇▇▇▇▇▇▇██████
val/mf1_scene,▁▅▃▅▇▆▆█▆▄▇▅▆▆▇▆▆▆▆▇▇▇▆▇▇
val/mf1_timeofday,▁▇▇▇▇▆▆▆▇▆▆▇▇▆▇▇▇▇▇█▇████
val/mf1_weather,▁▄▄▂▇▃▆█▆▇█▇▇▆▆▇▇▆▇█▇████
epoch,25
lr,0
train/loss,0.02435
val/avg_macro_f1,0.68597


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level2_vit_s16_pretrained.pth


## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.